In [2]:
import pandas as pd

In [3]:
import getpass

schema = "lianes_library"
host = "127.0.0.1"
user = "root"
password = getpass.getpass("MySQL-Passwort für 'root': ")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [4]:
authors_from_sql = pd.read_sql("authors", con=connection_string)
authors_from_sql

,authorID,first_name,last_name
0,1,Stephen,King
1,2,Joanne K.,Rowling
2,3,J.R.R.,Tolkien
3,4,George,Orwell
4,5,Jane,Austen
5,6,Dan,Brown
6,7,Agatha,Christie
7,8,Frank,Herbert
8,9,Suzanne,Collins
9,10,Michael,Ende


In [6]:
# Nur Titel und Genre aus der books-Tabelle abfragen,
# statt der kompletten Tabelle mit allen Spalten
pd.read_sql("""
            SELECT title, genre
            FROM books
            """,
            con=connection_string)

,title,genre
0,Es,Horror
1,Harry Potter und der Stein der Weisen,Fantasy
2,Der Hobbit,Fantasy
3,1984,Dystopie
4,Stolz und Vorurteil,Roman
5,Illuminati,Thriller
6,Mord im Orient-Express,Krimi
7,Dune,Science-Fiction
8,Die Tribute von Panem,Dystopie
9,Die unendliche Geschichte,Fantasy


In [7]:
# Ein neues Buch, das wir zur Datenbank hinzufügen wollen
neues_buch = pd.DataFrame({
    "AutorName": ["Stephen King"],
    "title": ["Shining"],
    "genre": ["Horror"]
})

neues_buch

,AutorName,title,genre
0,Stephen King,Shining,Horror


In [8]:
# Neues Buch direkt mit bekannter authorID (Stephen King = 1)
neues_buch = pd.DataFrame({
    "authorID": [1],
    "title": ["Shining"],
    "genre": ["Horror"]
})

In [9]:
# Diese Zeile sendet "neues_buch" tatsächlich an die MySQL-Datenbank.
# if_exists='append' heißt: bestehende Daten in der books-Tabelle bleiben
# unverändert, das neue Buch wird nur hinzugefügt (nicht überschrieben).
neues_buch.to_sql('books',
                   if_exists='append',
                   con=connection_string,
                   index=False)

1

In [10]:
# Für gezielte Änderungen braucht man "engine" statt nur den connection_string
from sqlalchemy import create_engine, text

engine = create_engine(connection_string)

# Der eigentliche SQL-Befehl: genau EIN Buch (bookID 11) ändern
update_query = """
    UPDATE books
    SET title = "The Shining"
    WHERE bookID = 11;
"""

# Transaction = eine "abgesicherte" Änderung: entweder klappt alles,
# oder (bei einem Fehler) wird automatisch nichts geändert (rollback)
with engine.connect() as connection:
    transaction = connection.begin()
    try:
        connection.execute(text(update_query))
        transaction.commit()
    except:
        transaction.rollback()
        raise

In [11]:
# Prüfen, ob die Änderung geklappt hat
pd.read_sql("""
            SELECT bookID, title
            FROM books
            WHERE bookID = 11
            """,
            con=connection_string)


,bookID,title
0,11,The Shining
